<img src="../../figs/holberton_logo.png" alt="logo" width="500"/>

# Transfer Learning

Humans have an inherent ability to transfer knowledge across tasks. What we acquire as knowledge while learning about one task, we utilize in the same way to solve related tasks. The more related the tasks, the easier it is for us to transfer, or cross-utilize our knowledge. Some simple examples would be,

- Know how to ride a motorbike ⮫ Learn how to ride a car
- Know how to play classic piano ⮫ Learn how to play jazz piano
- Know math and statistics ⮫ Learn machine learning

<img src="img/tlidea.webp" alt="logo" width="500"/>

In each of the above scenarios, we don’t learn everything from scratch when we attempt to learn new aspects or topics. We transfer and leverage our knowledge from what we have learnt in the past!


## Introduction

Conventional machine learning and deep learning algorithms, so far, have been traditionally designed to work in isolation. These algorithms are trained to solve specific tasks. The models have to be rebuilt from scratch once the feature-space distribution changes. **Transfer learning is the idea of overcoming the isolated learning paradigm and utilizing knowledge acquired for one task to solve related ones. **

### Intuition
Transfer learning is a technique where **a model developed for one task is reused as the starting point for a model on a second task**. This concept is especially useful in the domain of deep learning, where models are trained on large datasets. Instead of starting from scratch, transfer learning allows us to leverage the features learned by pre-trained models on large datasets (e.g., ImageNet) to solve related tasks with less data and computational resources.


The first thing to remember here is that, transfer learning, is not a new concept which is very specific to deep learning. There is a stark difference between the traditional approach of building and training machine learning models, and using a methodology following transfer learning principles.

<img src="img/tlframework.webp" alt="logo" width="700"/>


### Importance
The importance of transfer learning in the context of Convolutional Neural Networks (CNNs) is multifaceted:

- **Efficiency**: Reduces the computational cost and time required to train deep neural networks from scratch.
- **Performance**: Improves the performance of models by building on the rich, generalized features extracted by pre-trained networks.
- **Data Scarcity**: Especially beneficial when the task-specific data is limited, as the pre-trained models already possess valuable learned features.

### Applications

Transfer learning is widely applied across various domains:

- **Image Classification**: Fine-tuning pre-trained models for specific image classification challenges.
- **Object Detection**: Enhancing object detection frameworks with pre-trained backbone networks.
- **Image Segmentation**: Utilizing pre-trained encoders in image segmentation tasks.
- **Medical Imaging**: Adapting general models to specific tasks such as tumor detection or organ segmentation in medical images.
- **Natural Language Processing**: Leveraging pre-trained language models for specific tasks like sentiment analysis or machine translation.

## Technical Overview

### How Transfer Learning is Done

Transfer learning involves several key steps:

#### 1. **Select a Pre-trained Model**

**Selecting a pre-trained model is the first step in transfer learning**. These models, such as `VGG16`, `ResNet`, or `Inception`, have been trained on large datasets like `ImageNet`, which consists of millions of images across thousands of classes. 

The importance of this step lies in **leveraging the rich feature representations these models have already learned**. By starting with a pre-trained model, **we can avoid the need to learn low-level features from scratch**, which significantly reduces the amount of data and computation required. 

This practical aspect allows us to focus on fine-tuning the model for our specific task, saving both time and resources.

<img src="img/tldeep.webp" alt="logo" width="700"/>



#### 2. **Load the Model with Pre-trained Weights**: Load the pre-trained weights to initialize the model.

```python
from tensorflow.keras.applications import VGG16

# Load the VGG16 model pre-trained on ImageNet without the top (fully connected) layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Display the architecture of the base model
base_model.summary()
```


#### 3. **Freeze the Base Layers** 

Freezing the base layers of the pre-trained model is a crucial step in transfer learning. This means **setting these layers to non-trainable so that their weights, which have been optimized during the initial training on a large dataset, remain unchanged**. 

The importance of this step is to retain the valuable features that the model has already learned. These features often include fundamental patterns such as edges, textures, and shapes, which are transferable to many different tasks. 

Practically, **freezing the base layers helps to prevent overfitting**, especially when working with smaller datasets, as it reduces the number of parameters that need to be learned.


```python
# Freeze all layers in the base model to prevent their weights from being updated during training
for layer in base_model.layers:
    layer.trainable = False
```

#### 4. **Add Custom Layers**

After freezing the base layers, we **add custom layers to adapt the pre-trained model to our specific task**. These layers are typically fully connected layers that are tailored to the number of classes in our dataset. This step is crucial because it **allows the model to learn task-specific features while building on the generalized features** from the pre-trained layers. 

Practically, adding custom layers transforms a general-purpose model into a specialized one, capable of performing the new task efficiently. This customization enables the model to output predictions that are relevant to the specific problem we are trying to solve.


```python
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Add custom layers on top of the base model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)  # Assuming the task has 10 classes

# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Display the architecture of the final model
model.summary()
```
#### 5. **Compile and Train the Model**:

Compiling and training the model involves **specifying the loss function, optimizer, and metrics**, followed by training the model on the new dataset. This step is important because it integrates the new custom layers with the pre-trained base, enabling the model to adjust to the specifics of the new task. 


Practically, **this step allows us to fine-tune the network, ensuring that the new layers learn the necessary representations while the pre-trained layers provide a solid foundation**. This process significantly reduces training time and improves performance compared to training from scratch.
```python
from tensorflow.keras.optimizers import Adam

# Compile the model with a low learning rate
model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Assume 'train_generator' and 'validation_generator' are predefined data generators
model.fit(train_generator, epochs=10, validation_data=validation_generator)
```

#### 6. Fine-tuning Some of the Top Layers
Fine-tuning involves **unfreezing and retraining some of the top layers of the pre-trained model, often with a lower learning rate**. This step is essential for further improving the model's performance on the new task by allowing the top layers to adapt to the new data. 

The importance of this step lies in its ability to improve the model's ability to learn task-specific features while still leveraging the previously learned features from the pre-trained model. 

Practically, **fine-tuning strikes a balance between preserving useful pre-trained features and adapting the model to better fit the new data**, leading to better overall performance.

```python 
# Unfreeze the top 5 layers of the base model
for layer in base_model.layers[-5:]:
    layer.trainable = True

# Re-compile the model with a smaller learning rate
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Continue training the model
model.fit(train_generator, epochs=10, validation_data=validation_generator)
```

## Conclusion

Using transfer learning with CNNs and Keras Applications is a **highly effective way to use pre-trained models for different tasks**. By using transfer learning, you can **save a lot of time and computational power** needed to train deep neural networks from scratch, **while still getting high performance even if you don't have much data**. This technique is very useful for real-world problems like image classification and medical imaging, giving you a strong starting point for creating advanced models.

